In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

import math

import time

from time import sleep

import os

from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.support.ui import Select

from selenium.webdriver.support import expected_conditions

from webdriver_manager.chrome import ChromeDriverManager

# %%

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CA CIRO' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running CA CIRO Web Scraping Tool v.1.3


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




# %%

In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        'CA CIRO 1': 'https://www.ciro.ca/office-investor/dealers-we-regulate',

        }



Typology={

        'CA CIRO 1': 'List of Dealers We Regulate',


        }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [],
        }


now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')




In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



# Define a function to scroll to the bottom of the page

def scroll_to_bottom(driver):

    # Get scroll height

    last_height = driver.execute_script("return document.body.scrollHeight")



    while True:

        # Scroll down to the bottom

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        # Wait to load the page

        sleep(3)
        # Calculate new scroll height and compare with last scroll height

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:

            break

        last_height = new_height

def click_element_by_xpath(driver, xpath):

    # Find the element and click

    element = driver.find_element(By.XPATH, xpath)

    element.click()
    

def click_on_cookies(web_driver):

    try:

        web_driver.find_element(By.XPATH,f'//*[@id="modal-content-id-1"]/footer/div/button[3]').click()
        print('[Success] : Success to Click Cookie')

    except Exception as err:

        print('[ERROR] : Failed to click "I Accept" button on the cookies banner:', err)
        

def scrollinAndClick(xpath,key_press=False):
    if len(xpath) != 0 :
        for times in range(60):
            try:
                driver.find_element(By.XPATH, xpath).click()
                sleep(1)
                break
            except:
                print(f"[ERROR] : trying {times+1}/10 to key press 'DOWN' (scrolling)")
                sleep(1)
                if key_press:                    
                    driver.find_element(By.TAG_NAME, 'body').send_keys(key_press)
        else:   
            raise Exception(f'[ERROR] : Failed scrollin Or Click on xpath element : {xpath}')

def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)[0]}")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/{wait_time*2} s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )



In [6]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")


    driver.get(regdict[reg])    

    lastPage = False
    while not lastPage:
        soup=BeautifulSoup(driver.page_source, 'html.parser')

        sleep(3)

        nodes = soup.find_all('article',class_='node')
            
        for node in nodes:
            br_tags = node.find_all('br')
            if br_tags:
                for br in br_tags:
                    br.replace_with(' ')


            name = node.find('h2').find('span').text
            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)
            address = node.find('div',class_='clearfix').text
            sqldict['Address_1'].append(address)
            
            phone = node.find('div',class_='field--name-field-telephone')
            
            if phone is None:
                print(' ')
                sqldict['Phone'].append('')
            else:
                sqldict['Phone'].append(phone.text)
                print(phone.text)
            
            website = node.find('div',class_='field--name-field-website')
            
            if website is None:
                sqldict['Website'].append('')
                print(' ')
            else:
                sqldict['Website'].append(website.text)
                print(website.text)
            category = node.find('div',class_='field--name-field-registration-category').find('div',class_='field__item').text
            sqldict['Typology'].append(category)
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')

        try:
            scroll_to_bottom(driver)
            sleep(3)
            print(f"Current Page: {soup.find('ul',class_='pager__items').find('li',class_='pager__item is-active').text}")      
            sleep(3)
            href = soup.find('ul',class_='pager__items').find('li',class_='pager__item pager__item--next').find('a')['href']
            driver.get('https://www.ciro.ca/office-investor/dealers-we-regulate'+href)
            #next_page_li = driver.find_element_by_xpath("//a[@title='Next Page']")
            #next_page_li.click()
        except:
            lastButton = soup.find('ul',class_='pager__items').find('li',class_='pager__item pager__item--last ')
            if lastButton:
                pass
            else: 
                print('Failed to click the last page')
                lastPage = True
            
            
    


        

sqldict = bourange_same_length_array(sqldict)

[INFO] : Working 1/1 _(CA CIRO 1)_ 
905-597-5000
www.3ifinancial.com
416-274-5884
 
905-771-7338
 
416-777-9005
www.ackerfinley.com
403-571-0300
www.acumencapital.com
905-879-5075
 
+44 207 726 4003
www.admiralmarkets.com
604-687-1597
 
Current Page:    Current page 1 
1-855-462-4672
www.agoracorp.ca/
905-906-5288
www.aimstar.ca
519-364-6093
 
905-639-5115
www.alignedcapitalpartners.com
905-307-3688
www.amerity.ca/
905-709-7066
www.argosyne
604-434-3863
www.artechservices.ca/
514-935-9333
www.artoninvest.com
Current Page:    Current page 2 
416-348-9994
www.cifinancial.com/ci-assante/ca/en/index.html
416-364-1145
www.cifinancial.com/ci-assante/ca/en/index.html
1-888-282-3863
www.atb.com/personal
514-499-8440
www.auray.com
604-714-3900
 
416-964-0660
www.b2bbank.com/dealerservices/
416-964-0660
www.b2bbank.com
416-863-8900
www.barclays.com
Current Page:    Current page 3 
416-643-3830
www.beaconsecurities.ca
 
www.belaywealth.com/
416-640-7580
www.bloomburton.com
416-365-7161
www.bloomb

In [7]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df  = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
     

C:\Users\wuj1\AppData\Local\Temp\7\ipykernel_19652\2773079153.py:12: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [8]:

df.to_csv('total_ver5.csv')

In [9]:

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 4739 values.
Key 'priority' has 4739 values.
Key 'ListLabel' has 4739 values.
Key 'Typology' has 4739 values.
Key 'EntryType' has 4739 values.
Key 'Name' has 4739 values.
Key 'InternalID_1' has 4739 values.
Key 'InternalID_1_type' has 4739 values.
Key 'InternalID_2' has 4739 values.
Key 'InternalID_2_type' has 4739 values.
Key 'InternalID_3' has 4739 values.
Key 'InternalID_3_type' has 4739 values.
Key 'CoType' has 4739 values.
Key 'License_Type' has 4739 values.
Key 'Address_1' has 4739 values.
Key 'Address_2' has 4739 values.
Key 'City' has 4739 values.
Key 'Zip' has 4739 values.
Key 'Cntry' has 4739 values.
Key 'Phone' has 4739 values.
Key 'Fax' has 4739 values.
Key 'Website' has 4739 values.
Key 'Email' has 4739 values.
Key 'RegulationType' has 4739 values.
Key 'RegulationTypeCode' has 4739 values.
Key 'RegulationDate' has 4739 values.
Key 'CancellationDate' has 4739 values.
Key 'RegCtry' has 4739 values.
Key 'RegCode' has 4739 values.
Key 'ListCode' has 4739 values

Phone: 905-940-0094
Website: www.worldsourcesecurities.com
